In [1]:
import re, html, math
import nltk
import matplotlib.pyplot as plt
import gradio as gr
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

for p in ['punkt', 'punkt_tab', 'vader_lexicon', 'stopwords']:
    nltk.download(p, quiet=True)

sia = SentimentIntensityAnalyzer()
STOP = set(stopwords.words('english'))
esc = html.escape

loaded_words = {
    'slammed', 'blasted', 'disaster', 'disgrace', 'outrage', 'outrageous', 'shocking',
    'furious', 'radical', 'extremist', 'corrupt', 'fraud', 'scandal', 'lies', 'liar',
    'traitor', 'betrayal', 'tyranny', 'hypocrite', 'pathetic', 'ridiculous', 'stupid',
    'insane', 'terrible', 'horrible', 'evil', 'brutal', 'dangerous', 'nightmare',
    'devastating', 'unbelievable', 'incredible', 'heroic', 'brilliant', 'hate',
    'destroy', 'attack', 'crisis', 'failure', 'massive'
}

DISCLAIMER = (
    "**Important:** This report measures the tone of the wording "
    "(positive, negative or neutral) and how opinionated the writing sounds, "
    "using word-based methods. It does not measure political bias, factual "
    "accuracy or the writer's intent. A sad or negative topic can score "
    "negative even when the reporting is fair, so treat this as one signal "
    "to read alongside the article, not as a verdict."
)

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800&display=swap');
body, .gradio-container { background: linear-gradient(160deg, #e6f2ff 0%, #d7e9ff 45%, #e4f8ff 100%) !important; }
.gradio-container { font-family: 'Poppins', system-ui, -apple-system, 'Segoe UI', sans-serif !important; max-width: 1080px !important; margin: 0 auto !important; color: #1e3a5f; }
.hero { position: relative; overflow: hidden; background: linear-gradient(135deg, #d6ecff 0%, #8ec5fc 52%, #8fa8f8 100%); color: #12306b; border-radius: 30px; padding: 34px 38px; border: 2px solid rgba(255, 255, 255, .9); box-shadow: 0 16px 40px rgba(59, 130, 246, .25); margin: 6px 0 14px; }
.hero::before { content: ""; position: absolute; width: 230px; height: 230px; right: -50px; top: -70px; background: radial-gradient(circle, rgba(255, 255, 255, .9), rgba(255, 255, 255, 0) 70%); }
.hero::after { content: ""; position: absolute; width: 200px; height: 200px; left: 42%; bottom: -100px; background: radial-gradient(circle, rgba(125, 211, 252, .75), rgba(125, 211, 252, 0) 70%); }
.hero > * { position: relative; z-index: 1; }
.hero h1 { margin: 12px 0 8px; font-size: 2.15rem; font-weight: 800; letter-spacing: -.02em; color: #0f2a5c; line-height: 1.15; }
.hero p { margin: 0 0 16px; max-width: 740px; font-size: 1.02rem; line-height: 1.65; color: #1e3a5f; }
.hero-badge { display: inline-block; background: rgba(255, 255, 255, .8); color: #3b82f6; padding: 5px 14px; border-radius: 999px; font-size: .78rem; font-weight: 700; letter-spacing: .06em; text-transform: uppercase; }
.pills span { display: inline-block; border: 1px solid rgba(59, 130, 246, .3); color: #1e40af; padding: 6px 14px; border-radius: 999px; margin: 0 8px 8px 0; font-size: .86rem; font-weight: 600; background: rgba(255, 255, 255, .7); }
.pills span:nth-child(1) { background: #e0f2fe; }
.pills span:nth-child(2) { background: #dbeafe; }
.pills span:nth-child(3) { background: #cffafe; }
.pills span:nth-child(4) { background: #e0e7ff; }
.pills span:nth-child(5) { background: #c7d2fe; }
button[role="tab"] { font-weight: 600 !important; font-size: .98rem !important; }
button[role="tab"][aria-selected="true"] { color: #2563eb !important; border-color: #38bdf8 !important; }
.card { background: rgba(255, 255, 255, .78); color: #1e3a5f; border: 2px solid #dbeafe; border-radius: 26px; padding: 18px 20px; box-shadow: 0 10px 30px rgba(59, 130, 246, .14); margin-bottom: 16px; }
textarea, input[type="text"] { border-radius: 16px !important; }
textarea:focus, input[type="text"]:focus { border-color: #93c5fd !important; box-shadow: 0 0 0 4px rgba(147, 197, 253, .4) !important; }
.main-btn { background: linear-gradient(135deg, #38bdf8 0%, #3b82f6 55%, #6366f1 100%) !important; color: #ffffff !important; border: 2px solid rgba(255, 255, 255, .9) !important; border-radius: 999px !important; font-weight: 700 !important; font-size: 1rem !important; padding: 12px 0 !important; box-shadow: 0 10px 22px rgba(59, 130, 246, .42) !important; }
.main-btn:hover { filter: brightness(1.04); transform: translateY(-2px); }
.verdict-wrap { display: flex; gap: 14px; flex-wrap: wrap; align-items: stretch; margin-bottom: 16px; }
.verdict { flex: 0 0 250px; border-radius: 24px; padding: 20px 22px; border: 2px solid rgba(255, 255, 255, .85); box-shadow: 0 10px 24px rgba(59, 130, 246, .2); }
.verdict.positive { background: linear-gradient(135deg, #d4fae6, #a7f3d0); color: #14532d; }
.verdict.negative { background: linear-gradient(135deg, #ffe4ea, #fecdd3); color: #881337; }
.verdict.neutral { background: linear-gradient(135deg, #eaf1f9, #cddbed); color: #2c4a6e; }
.v-label { font-size: .74rem; font-weight: 700; letter-spacing: .08em; text-transform: uppercase; opacity: .75; }
.v-main { font-size: 1.8rem; font-weight: 800; letter-spacing: -.01em; margin: 4px 0 2px; }
.v-sub { font-size: .86rem; opacity: .85; }
.verdict.small { flex: none; padding: 14px 16px; box-shadow: none; margin-bottom: 12px; }
.verdict.small .v-main { font-size: 1.4rem; }
.tiles { flex: 1 1 320px; display: flex; flex-wrap: wrap; gap: 10px; }
.tile { flex: 1 1 140px; border: 2px solid rgba(255, 255, 255, .9); border-radius: 18px; padding: 12px 14px; background: #eef5ff; color: #1e3a5f; }
.tile:nth-child(1) { background: #e3f1ff; }
.tile:nth-child(2) { background: #e7eeff; }
.tile:nth-child(3) { background: #e1f6fb; }
.tile:nth-child(4) { background: #eaf3ff; }
.tile .k { font-size: .7rem; font-weight: 700; text-transform: uppercase; letter-spacing: .06em; opacity: .65; }
.tile .v { font-size: 1.05rem; font-weight: 700; margin-top: 3px; }
.bar-title { font-size: .78rem; font-weight: 700; text-transform: uppercase; letter-spacing: .06em; opacity: .7; margin: 4px 0 6px; }
.bar { display: flex; height: 16px; border-radius: 999px; overflow: hidden; background: #e3eefb; }
.seg.pos { background: #6ee7b7; }
.seg.neu { background: #c7d7ee; }
.seg.neg { background: #fda4af; }
.legend { display: flex; gap: 18px; flex-wrap: wrap; margin-top: 8px; font-size: .86rem; font-weight: 500; }
.dot { display: inline-block; width: 11px; height: 11px; border-radius: 50%; margin-right: 6px; }
.dot.pos { background: #6ee7b7; }
.dot.neu { background: #c7d7ee; }
.dot.neg { background: #fda4af; }
.chip { display: inline-block; padding: 3px 12px; border-radius: 999px; font-size: .78rem; font-weight: 700; letter-spacing: .03em; }
.chip.positive { background: #c9f7e1; color: #166534; }
.chip.negative { background: #ffd9e0; color: #9f1239; }
.chip.neutral { background: #dde7f5; color: #334e68; }
.cmp-grid { display: flex; gap: 14px; flex-wrap: wrap; margin-bottom: 14px; }
.cmp-card { flex: 1 1 300px; border: 2px solid #dbeafe; border-radius: 24px; padding: 16px 18px; background: rgba(255, 255, 255, .85); box-shadow: 0 8px 24px rgba(59, 130, 246, .14); color: #1e3a5f; }
.cmp-title { font-weight: 800; font-size: 1.05rem; margin-bottom: 10px; color: #1d4ed8; }
.summary { background: #e0f0ff; border: 2px solid #93c5fd; color: #1e3a8a; border-radius: 18px; padding: 12px 16px; font-weight: 600; margin-bottom: 14px; }
.cmp-table { width: 100%; border-collapse: collapse; font-size: .93rem; color: #1e3a5f; }
.cmp-table th { text-align: left; font-size: .72rem; text-transform: uppercase; letter-spacing: .06em; opacity: .7; padding: 8px 10px; border-bottom: 2px solid #dbeafe; }
.cmp-table td { padding: 10px; border-bottom: 1px solid #dbeafe; vertical-align: top; }
.cmp-table td:first-child { font-weight: 600; width: 28%; }
.msg { border: 2px dashed #93c5fd; border-radius: 18px; padding: 16px 18px; background: #eff6ff; color: #1d4ed8; font-weight: 600; }
.note { border-left: 5px solid #38bdf8 !important; background: #eef6ff !important; color: #1e3a5f !important; border-radius: 16px !important; padding: 12px 16px !important; font-size: .9rem; }
.foot { text-align: center; font-size: .82rem; color: #6b8bb5; margin: 6px 0 14px; }
.doc table { width: 100%; }
.seg.fact { background: #a5d8ff; }
.seg.mix { background: #fde68a; }
.seg.opi { background: #fdba74; }
.dot.fact { background: #a5d8ff; }
.dot.mix { background: #fde68a; }
.dot.opi { background: #fdba74; }
.ev-title { font-size: .78rem; font-weight: 700; text-transform: uppercase; letter-spacing: .06em; opacity: .7; margin: 4px 0 4px; }
.pie-box { margin-top: 10px; }
.f-radio { display: none; }
.pie-row { display: flex; gap: 30px; align-items: center; flex-wrap: wrap; margin: 10px 0 14px; }
.pie { position: relative; width: 230px; height: 230px; flex: 0 0 230px; }
.wedge { position: absolute; inset: 0; cursor: pointer; transform-origin: 50% 50%; transition: transform .2s, filter .2s; }
.wedge:hover { transform: scale(1.04); filter: brightness(.96); }
.pie-hole { position: absolute; inset: 28%; border-radius: 50%; background: #f8fbff; display: flex; flex-direction: column; align-items: center; justify-content: center; pointer-events: none; box-shadow: inset 0 2px 10px rgba(59, 130, 246, .2); color: #1e3a5f; }
.pie-hole b { font-size: 1.7rem; font-weight: 800; line-height: 1.1; }
.pie-hole span { font-size: .7rem; text-transform: uppercase; letter-spacing: .06em; opacity: .7; }
.pie-legend { display: flex; flex-direction: column; gap: 10px; min-width: 250px; }
.lg { cursor: pointer; display: flex; align-items: center; gap: 8px; padding: 10px 16px; border-radius: 16px; border: 2px solid transparent; background: #eff6ff; font-size: .95rem; color: #1e3a5f; }
.lg small { margin-left: auto; opacity: .55; font-size: .72rem; }
.lg:hover { border-color: #bfdbfe; }
.panel { display: none; margin-top: 4px; }
.panel-title { font-weight: 700; margin: 4px 0 6px; color: #1d4ed8; }
.sent-panels { max-height: 380px; overflow-y: auto; }
#f-pos:checked ~ .sent-panels .panel-pos, #f-neu:checked ~ .sent-panels .panel-neu, #f-neg:checked ~ .sent-panels .panel-neg { display: block; }
#f-pos:checked ~ .pie-row .w-pos, #f-neu:checked ~ .pie-row .w-neu, #f-neg:checked ~ .pie-row .w-neg { transform: scale(1.07); filter: drop-shadow(0 6px 10px rgba(80, 60, 140, .25)); }
#f-pos:checked ~ .pie-row .lg-pos { border-color: #6ee7b7; background: #ecfdf5; }
#f-neu:checked ~ .pie-row .lg-neu { border-color: #93a9c9; background: #eef3fa; }
#f-neg:checked ~ .pie-row .lg-neg { border-color: #fda4af; background: #fff1f2; }
.sent-card { border-radius: 16px; padding: 10px 14px; margin: 8px 0; font-size: .93rem; line-height: 1.55; color: #1e3a5f; }
.sent-card.pos { background: #ecfdf5; border: 1px solid #bbf7d0; }
.sent-card.neg { background: #fff1f2; border: 1px solid #fecdd3; }
.sent-card.neu { background: #eef3fa; border: 1px solid #cbd9ec; }
.sent-card .sc { float: right; font-size: .72rem; font-weight: 700; opacity: .6; margin-left: 10px; }
.sent-empty { opacity: .65; padding: 8px 2px; }
.pie-hint { font-size: .82rem; opacity: .65; }
.dark .pie-hole { background: #14305a; color: #eaf2ff; }
.dark .lg, .dark .sent-card { background: rgba(30, 58, 100, .55) !important; color: #eaf2ff !important; }
.sent-card.fact { background: #eaf4ff; border: 1px solid #bfdbfe; }
.sent-card.mix { background: #fffbe6; border: 1px solid #fde68a; }
.sent-card.opi { background: #fff3e6; border: 1px solid #fed7aa; }
.tag { display: inline-block; padding: 2px 10px; border-radius: 999px; font-size: .7rem; font-weight: 800; letter-spacing: .04em; text-transform: uppercase; margin-right: 8px; }
.tag.fact { background: #cfe6ff; color: #1e4e8c; }
.tag.mix { background: #fdecab; color: #7a5a00; }
.tag.opi { background: #ffd9b0; color: #8a4300; }
.topics { display: flex; flex-wrap: wrap; gap: 8px; margin: 2px 0 14px; }
.topic { padding: 6px 14px; border-radius: 999px; font-size: .88rem; font-weight: 600; color: #1e3a5f; border: 2px solid rgba(255, 255, 255, .9); }
.topic.t0 { background: #e0f2fe; }
.topic.t1 { background: #dbeafe; }
.topic.t2 { background: #cffafe; }
.topic.t3 { background: #e0e7ff; }
.topic.t4 { background: #c7d2fe; }
.verdict.fact { background: linear-gradient(135deg, #e6f2ff, #bfdbfe); color: #1e3a8a; }
.verdict.mix { background: linear-gradient(135deg, #fff8d6, #fde68a); color: #78350f; }
.verdict.opi { background: linear-gradient(135deg, #ffedd5, #fed7aa); color: #7c2d12; }
.cmp-head { font-size: .92rem; font-style: italic; opacity: .85; line-height: 1.4; margin: -4px 0 10px; }
.dark body, .dark .gradio-container { background: linear-gradient(160deg, #0d1b3a 0%, #10284f 50%, #0c2a45 100%) !important; }
.dark .card, .dark .cmp-card, .dark .tile { background: rgba(30, 58, 100, .55) !important; color: #eaf2ff !important; border-color: #2b4c80 !important; }
.dark .cmp-table, .dark .cmp-title { color: #eaf2ff !important; }
"""

HERO = """
<div class="hero">
  <span class="hero-badge">&#10024; vibe check for the news</span>
  <h1>News Sentiment &amp; Style Analyzer</h1>
  <p>Paste a news article and see how it feels: the tone sentence by sentence, how opinionated the writing is,
  its strongly worded words, and how a chosen name is written about.</p>
  <div class="pills"><span>&#128172; Tone</span><span>&#127912; Style</span><span>&#128273; Key points</span><span>&#128269; Name tracking</span><span>&#9878;&#65039; Compare two articles</span></div>
</div>
"""

HEAD_HTML = "<style>" + CSS + "</style>" + HERO

FOOT_HTML = '<div class="foot">Built with NLTK VADER, TextBlob, Matplotlib and Gradio &#10024;</div>'

MSG_SHORT = '<div class="msg">Please paste a full article (at least 30 words).</div>'
MSG_SHORT_BOTH = '<div class="msg">Please paste both articles (at least 30 words each).</div>'

WRITEUP = """
# Project Write-up: News Sentiment and Style Analyzer

*Prepared by [your name], [course / institution], November 2026*

## Abstract
This project builds a tool that reads a news article and reports how it sounds: whether its wording is positive, negative or neutral, how the tone changes from sentence to sentence, how opinionated the writing is, and which strongly worded words it uses. It can track the tone of sentences that mention a chosen person, party or organization, and it can compare two articles side by side. It was built and tested on 870 English-language Indian news articles. The project deliberately does not claim to detect political bias, because no reliable bias labels were available and because tone and bias are different things.

## 1. Problem and objectives
'Media bias' is hard to measure: people disagree about what counts as biased, labeled data is scarce, and one article rarely shows bias on its own. The project therefore asks a narrower question: what can language signals (tone, subjectivity, strongly worded words) tell a reader about an article, and where do they mislead? Objectives: (1) a plain-language sentiment report for any English article, (2) sentence-level tone, (3) tone around a named entity, (4) side-by-side comparison of two articles, (5) honest testing and clear limitations.

## 2. Data
**Reference set:** a public Kaggle dataset of Indian news (1,568 articles from Firstpost and NDTV; columns source, category, link, author, published_at, header, subheader, content). Filtered to the india and world categories, and articles under 50 words removed, leaving **870 articles** (528 Firstpost, 342 NDTV), mostly from 2020.

**Exploratory set:** the Hyperpartisan News Detection dataset (SemEval-2019 Task 4, 645 US articles from 2017), used for an early experiment described below.

## 3. Method
1. **Cleaning:** HTML tags removed, character codes decoded, spaces collapsed.
2. **Sentence tone:** text split into sentences (NLTK) and each scored with VADER, a rule-based sentiment tool (-1 very negative to +1 very positive). Scores of +0.05 or more are positive, -0.05 or less are negative, the rest neutral. The overall label uses the average.
3. **Style:** TextBlob subjectivity (0 factual, 1 opinion) shown as Mostly factual (below 0.30), A mix of facts and opinion (0.30 to 0.50) or Opinion-heavy (above 0.50).
4. **Strongly worded words:** a hand-made list of 40 emotionally charged words is matched against the article.
5. **Summary and key topics:** an extractive summary picks the three sentences whose words appear most often across the article and shows them in their original order. Nothing is rewritten, so it cannot add facts. The most frequent content words are shown separately as key topics.
6. **Tone around a name:** sentences containing the chosen name (whole-word match, any case) are scored separately.
7. **Facts and opinion:** every sentence also gets a TextBlob subjectivity score. Below 0.30 counts as a fact sentence, 0.50 or above as an opinion sentence and anything in between as mixed.
8. **Interactive pie chart:** the share of positive, neutral and negative sentences is shown as a pie chart. Clicking a slice lists the sentences in that group, so the reader only reads them if they want to.
9. **Comparison:** two articles are shown side by side with their headlines, writing style, the share of fact, mixed and opinion sentences, and one example sentence each for fact, mixed and opinion.

## 4. Results and comparison

### 4.1 Reference statistics (870 Indian news articles)
| Measure | Mean | Notes |
|---|---|---|
| Article sentiment | 0.010 | Close to neutral on average |
| Headline sentiment | -0.056 | Headlines lean slightly negative |
| Strongly worded words per 1,000 words | 1.188 | Median is 0; top quarter above 1.9 |
| Subjectivity | 0.394 | Standard deviation 0.093 |

### 4.2 Comparison of the two outlets
| Outlet | Articles | Article sentiment | Headline sentiment | Strongly worded words per 1,000 | Subjectivity |
|---|---|---|---|---|---|
| Firstpost | 528 | 0.013 | -0.047 | 1.235 | 0.391 |
| NDTV | 342 | 0.006 | -0.070 | 1.115 | 0.397 |

The differences are very small on every measure. With two outlets, an uncontrolled mix of topics and a hand-made word list, these results do not support any claim that one outlet is more biased than the other. They mainly show a similar writing register on these measures.

### 4.3 Early experiment on US data (not adopted)
A combined score from tone, strongly worded words and subjectivity separated hyperpartisan from other articles only slightly better than chance (ROC AUC 0.577, where 0.5 is chance). A text classifier (TF-IDF with logistic regression) reached 72% accuracy and an AUC of 0.791 on 129 held-out articles, but the words it relied on included event and topic names (hurricane, Harvey, Weinstein, Florida) and outlet boilerplate (fox, associated, daily). It had partly learned which stories and sources were in the data, not how biased language sounds, so it was not used for Indian news.

### 4.4 What the tool gets wrong
On a 251-word test article the overall tone was neutral (+0.01) with subjectivity 0.40 and no strongly worded words, yet its most negative sentence was the plain fact 'He died in 2020.' (-0.56), because 'died' is a negative word. Likewise, a quote announcing a crackdown on crimes such as rape and murder is full of negative words and can score negative even when the speaker supports the action. Tone is not stance.

## 5. Ethics and responsible use
A tool that labels articles can be misused to pressure outlets, profile writers or justify censorship. This project presents results as language signals, states that they are not a verdict on bias or accuracy, and shows the sentences behind each score. It uses only public news text and stores no personal data. The tone-around-a-name feature describes how a name is written about, not whether the article is fair to it.

## 6. Future work
- Collect a multi-outlet Indian news set with human-labeled stance or bias and validate the tool against it.
- Add stance detection toward a named topic.
- Replace the word-based sentiment step with a transformer model and compare on human-labeled sentences.
- Extend to Hindi and Telugu.
- Grow and validate the strongly worded word list on Indian news text.

## 7. Conclusion
The project delivers a working interactive tool that explains how a news article sounds, shows where the tone shifts, tracks tone around a name and compares two articles. It also shows, with examples and numbers, why tone is not bias and why such a tool must be read as one signal among several.

## Tools and references
NLTK and VADER (Hutto and Gilbert, 2014, VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text, ICWSM); TextBlob; pandas; matplotlib; scikit-learn; Gradio; Kiesel et al. (2019), SemEval-2019 Task 4: Hyperpartisan News Detection; Kaggle dataset of Indian news articles (Firstpost and NDTV).
"""

LIMITATIONS = """
# Limitations

- **Tone is not stance.** The tool measures how the words sound, not whether an article supports or opposes something. A report on a crime scores negative even if it supports a tough response.
- **No context.** It cannot understand sarcasm, quotation or background. A quoted insult counts as the article's own language, and a plain fact like 'he died' counts as negative.
- **Small hand-made word list.** The strongly worded word list has 40 entries chosen by the author and has not been validated.
- **No human bias labels.** No labeled dataset of Indian news bias could be obtained, so the tool has not been validated against human judgments. Its output should not be called a bias score.
- **Narrow data.** The reference set has two outlets, mostly 2020 articles and a limited range of topics, so comparisons between outlets are weak evidence.
- **Tone around a name is approximate.** It scores the sentences that contain the name. An event described in such a sentence (for example an arrest) affects the score even if the article treats the person fairly.
- **Fact versus opinion is approximate.** Sentence labels use subjectivity thresholds from a word-based tool, so a factual sentence with emotional words can be labeled opinion, and an opinion written in plain words can be labeled fact.
- **English only.** The tools used do not handle Hindi, Telugu or other Indian languages.
- **Summaries are selections.** Choosing which sentences to show is itself a small editorial choice.
"""


def clean(t):
    t = re.sub(r'<[^>]+>', ' ', str(t))
    t = html.unescape(t)
    return re.sub(r'\s+', ' ', t).strip()


def label(x):
    if x >= 0.05:
        return "POSITIVE"
    if x <= -0.05:
        return "NEGATIVE"
    return "NEUTRAL"


def key_points(sents, n=3):
    if len(sents) <= n:
        return list(sents)
    freq = {}
    for s in sents:
        for w in re.findall(r"[a-z']+", s.lower()):
            if w not in STOP and len(w) > 2:
                freq[w] = freq.get(w, 0) + 1
    if not freq:
        return list(sents[:n])
    top = max(freq.values())
    freq = {w: c / top for w, c in freq.items()}
    ranked = []
    for i, s in enumerate(sents):
        ws = [w for w in re.findall(r"[a-z']+", s.lower()) if w in freq]
        length = len(s.split())
        if len(ws) < 4 or length > 60:
            continue
        ranked.append((sum(freq[w] for w in ws) / length, i))
    if not ranked:
        return list(sents[:n])
    best = sorted(sorted(ranked, reverse=True)[:n], key=lambda x: x[1])
    return [sents[i] for _, i in best]


def name_patterns(name):
    names = [n.strip() for n in str(name).split(',') if n.strip()]
    return [re.compile(r'\b' + re.escape(n) + r'\b', re.I) for n in names]


GENERIC = {'said', 'also', 'would', 'could', 'will', 'may', 'told', 'says', 'say', 'one', 'two', 'new',
           'year', 'years', 'time', 'like', 'many', 'much', 'made', 'make', 'including', 'according', 'since',
           'still', 'even', 'just', 'last', 'first', 'been', 'have', 'were', 'their', 'there', 'about', 'after',
           'which', 'while', 'where', 'other', 'than', 'into', 'over', 'more', 'most', 'some', 'only', 'what',
           'when', 'them', 'they', 'this', 'that', 'with', 'from', 'were', 'being', 'then', 'had', 'has'}


def key_topics(words, n=6):
    freq = {}
    for w in words:
        w = w.strip("'")
        if len(w) >= 4 and w.isalpha() and w not in STOP and w not in GENERIC:
            freq[w] = freq.get(w, 0) + 1
    ranked = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    return [w for w, c in ranked[:n] if c >= 2] or [w for w, c in ranked[:n]]


def core(title, text, name):
    text = clean(text)
    words = re.findall(r"[a-z']+", text.lower())
    if len(words) < 30:
        return None
    sents = sent_tokenize(text)
    scores = [sia.polarity_scores(s)['compound'] for s in sents]
    labels = [label(s) for s in scores]
    subjs = [TextBlob(s).sentiment.subjectivity for s in sents]
    kinds = ['opi' if v >= 0.5 else 'fact' if v < 0.3 else 'mix' for v in subjs]
    n = max(len(scores), 1)
    avg = sum(scores) / n
    subj = TextBlob(text).sentiment.subjectivity
    if subj < 0.30:
        opinion = "Mostly factual"
    elif subj < 0.50:
        opinion = "Facts and opinion mixed"
    else:
        opinion = "Opinion-heavy"
    pats = name_patterns(name)
    mentions = [i for i, s in enumerate(sents) if pats and any(p.search(s) for p in pats)]
    return dict(
        title=title, sents=sents, scores=scores, labels=labels, subjs=subjs, n=n, avg=avg,
        overall=label(avg), subj=subj, opinion=opinion, mentions=mentions,
        headline=label(sia.polarity_scores(title)['compound']) if title.strip() else None,
        found=sorted({w for w in words if w in loaded_words}),
        points=key_points(sents, 3),
        pos=labels.count("POSITIVE"), neg=labels.count("NEGATIVE"), neu=labels.count("NEUTRAL"),
        fact=kinds.count('fact'), mix=kinds.count('mix'), opi=kinds.count('opi'),
        kinds=kinds,
        topics=key_topics(words),
    )


def chip(lab):
    return f'<span class="chip {lab.lower()}">{lab}</span>'


def tile(k, v):
    return f'<div class="tile"><div class="k">{k}</div><div class="v">{v}</div></div>'


def name_summary(r):
    idx = r['mentions']
    if not idx:
        return "not mentioned"
    k = [r['kinds'][i] for i in idx]
    plural = 's' if len(idx) != 1 else ''
    return f"{len(idx)} sentence{plural}: {k.count('fact')} fact, {k.count('mix')} mixed, {k.count('opi')} opinion"


def name_section(r, name):
    name = name.strip()
    if not name:
        return ""
    out = f'### Tone around "{name}"\n\n'
    idx = r['mentions']
    if not idx:
        return out + "Not mentioned in this article.\n\n"
    sc = [r['scores'][i] for i in idx]
    avg = sum(sc) / len(sc)
    out += f"Mentioned in {len(idx)} of {r['n']} sentences. Tone of those sentences: **{label(avg)}** ({avg:+.2f}).\n\n"
    p = sum(s >= 0.05 for s in sc)
    g = sum(s <= -0.05 for s in sc)
    out += f"- Positive: {p}\n- Neutral: {len(sc) - p - g}\n- Negative: {g}\n\n"
    for i in idx[:5]:
        out += f"- [{r['labels'][i]}] {r['sents'][i][:200]}\n"
    return out + "\n"


EMO = {"POSITIVE": "\U0001F60A", "NEGATIVE": "\U0001F61F", "NEUTRAL": "\U0001F610"}
PIE_COL = {'pos': '#6ee7b7', 'neu': '#c7d7ee', 'neg': '#fda4af'}


def verdict_html(r):
    cls = r['overall'].lower()
    tiles = (
        tile("Headline", chip(r['headline']) if r['headline'] else "n/a")
        + tile("Style", r['opinion'])
        + tile("Sentences", r['n'])
        + tile("Strongly worded words", len(r['found']))
    )
    return (
        '<div class="verdict-wrap">'
        f'<div class="verdict {cls}"><div class="v-label">Overall sentiment</div>'
        f'<div class="v-main">{EMO[r["overall"]]} {r["overall"]}</div>'
        f'<div class="v-sub">Average tone {r["avg"]:+.2f} on a scale from -1 to +1</div></div>'
        f'<div class="tiles">{tiles}</div></div>'
    )


def topics_html(r):
    if not r['topics']:
        return ''
    chips = ''.join(f'<span class="topic t{i % 5}">{esc(w)}</span>' for i, w in enumerate(r['topics']))
    return f'<div class="bar-title">Key topics</div><div class="topics">{chips}</div>'


def wedge_clip(a0, a1):
    pts = ['50% 50%']
    steps = max(2, int((a1 - a0) * 120))
    for i in range(steps + 1):
        t = a0 + (a1 - a0) * i / steps
        ang = 2 * math.pi * t - math.pi / 2
        pts.append(f'{50 + 50 * math.cos(ang):.2f}% {50 + 50 * math.sin(ang):.2f}%')
    return 'polygon(' + ', '.join(pts) + ')'


def pie_html(r):
    n = r['n']
    groups = [('pos', 'Positive', 'POSITIVE'), ('neu', 'Neutral', 'NEUTRAL'), ('neg', 'Negative', 'NEGATIVE')]
    out = '<div class="pie-box">'
    for k, _, _ in groups:
        out += f'<input type="radio" name="sentfilter" id="f-{k}" class="f-radio">'
    out += '<div class="pie-row"><div class="pie">'
    start = 0.0
    for k, name, _ in groups:
        cnt = r[k]
        if cnt:
            frac = cnt / n
            clip = wedge_clip(start, start + frac)
            out += (f'<label for="f-{k}" class="wedge w-{k}" title="{name}: {cnt} of {n} sentences" '
                    f'style="background:{PIE_COL[k]};clip-path:{clip};-webkit-clip-path:{clip}"></label>')
            start += frac
    out += f'<div class="pie-hole"><b>{n}</b><span>sentences</span></div></div>'
    out += '<div class="pie-legend">'
    for k, name, _ in groups:
        cnt = r[k]
        out += (f'<label for="f-{k}" class="lg lg-{k}"><i class="dot {k}"></i>'
                f'<b>{name}</b>&nbsp;{cnt / n * 100:.0f}% ({cnt})<small>click to read</small></label>')
    out += '</div></div><div class="sent-panels">'
    for k, name, lab in groups:
        items = [(s, sc) for s, sc, l in zip(r['sents'], r['scores'], r['labels']) if l == lab]
        body = ''.join(
            f'<div class="sent-card {k}">{esc(s)}<span class="sc">{sc:+.2f}</span></div>' for s, sc in items
        ) or '<div class="sent-empty">No sentences in this group.</div>'
        out += f'<div class="panel panel-{k}"><div class="panel-title">{name} sentences ({len(items)})</div>{body}</div>'
    out += '</div><div class="pie-hint">Click a slice of the pie, or its label, to read those sentences.</div></div>'
    return out


def fo_bar(r):
    n = r['n']
    f, m, o = r['fact'] / n * 100, r['mix'] / n * 100, r['opi'] / n * 100
    return (
        f'<div class="bar"><span class="seg fact" style="width:{f:.1f}%"></span>'
        f'<span class="seg mix" style="width:{m:.1f}%"></span>'
        f'<span class="seg opi" style="width:{o:.1f}%"></span></div>'
        f'<div class="legend"><span><i class="dot fact"></i>Facts {f:.0f}%</span>'
        f'<span><i class="dot mix"></i>Mixed {m:.0f}%</span>'
        f'<span><i class="dot opi"></i>Opinion {o:.0f}%</span></div>'
    )


def evidence_html(r):
    sents, subjs, kinds = r['sents'], r['subjs'], r['kinds']
    out = '<div class="ev-title">Example sentences behind the split</div>'
    for k, name in (('fact', 'Fact'), ('mix', 'Mixed'), ('opi', 'Opinion')):
        idx = [i for i, s in enumerate(sents) if kinds[i] == k and 6 <= len(s.split()) <= 60]
        if not idx:
            idx = [i for i in range(len(sents)) if kinds[i] == k]
        if not idx:
            out += f'<div class="sent-empty"><span class="tag {k}">{name}</span>No {name.lower()} sentences in this article.</div>'
            continue
        if k == 'fact':
            i = min(idx, key=lambda j: (subjs[j], -len(sents[j])))
        elif k == 'opi':
            i = max(idx, key=lambda j: (subjs[j], len(sents[j])))
        else:
            i = min(idx, key=lambda j: abs(subjs[j] - 0.4))
        out += (f'<div class="sent-card {k}"><span class="tag {k}">{name}</span>{esc(sents[i])}'
                f'<span class="sc">subjectivity {subjs[i]:.2f}</span></div>')
    return out


STYLE_CLS = {
    "Mostly factual": ("fact", "\U0001F4F0"),
    "Facts and opinion mixed": ("mix", "\u2696\ufe0f"),
    "Opinion-heavy": ("opi", "\U0001F4AD"),
}


def card_html(r, title, headline=""):
    cls, emo = STYLE_CLS[r['opinion']]
    op = r['opi'] / r['n'] * 100
    head = f'<div class="cmp-head">{esc(headline.strip())}</div>' if headline.strip() else ''
    return (
        f'<div class="cmp-card"><div class="cmp-title">{title}</div>{head}'
        f'<div class="verdict small {cls}"><div class="v-main">{emo} {r["opinion"]}</div>'
        f'<div class="v-sub">{op:.0f}% of sentences are opinion</div></div>'
        f'<div class="bar-title">Facts and opinion</div>{fo_bar(r)}'
        f'<div style="height:12px"></div>{evidence_html(r)}</div>'
    )


def table_html(rows):
    out = ('<table class="cmp-table"><thead><tr><th></th><th>Article A</th>'
           '<th>Article B</th></tr></thead><tbody>')
    for k, x, y in rows:
        out += f'<tr><td>{k}</td><td>{x}</td><td>{y}</td></tr>'
    return out + '</tbody></table>'


POS_C, NEG_C, NEU_C, ACC_C = '#6ee7b7', '#fda4af', '#c7d7ee', '#3b82f6'
BG_C, TXT_C = '#f8fbff', '#3d5a80'


def draw(ax, r, title=None):
    colors = [POS_C if s >= 0.05 else NEG_C if s <= -0.05 else NEU_C for s in r['scores']]
    edges = [ACC_C if i in r['mentions'] else 'none' for i in range(len(r['scores']))]
    ax.figure.patch.set_facecolor(BG_C)
    ax.set_facecolor(BG_C)
    ax.bar(range(1, len(r['scores']) + 1), r['scores'], color=colors,
           edgecolor=edges, linewidth=2 if r['mentions'] else 0)
    ax.axhline(0, color='#9db4d3', linewidth=0.9)
    ax.set_ylim(-1, 1)
    ax.grid(axis='y', color='#e3eefb', linewidth=1)
    ax.set_axisbelow(True)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'):
        ax.spines[sp].set_color('#cfe0f5')
    ax.tick_params(colors=TXT_C)
    ax.set_xlabel('Sentence number', color=TXT_C)
    ax.set_ylabel('Tone score', color=TXT_C)
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold', color='#0f2a5c', loc='left')


def report_md(r, name):
    found = ', '.join(r['found']) if r['found'] else 'none'
    out = f"**Strongly worded words found:** {found}\n\n"
    out += name_section(r, name)
    out += "### Summary\n\n"
    out += " ".join(r['points']) + "\n"
    return out


def single(title, text, name):
    r = core(title, text, name)
    if r is None:
        return MSG_SHORT, None, ""
    fig, ax = plt.subplots(figsize=(8, 3))
    note = "   (outlined bars mention the tracked name)" if r['mentions'] else ""
    draw(ax, r, "How the tone changes through the article" + note)
    fig.tight_layout()
    plt.close(fig)
    return verdict_html(r) + topics_html(r) + pie_html(r), fig, report_md(r, name)


def headline_style(t):
    t = t.strip()
    if not t:
        return "n/a"
    v = TextBlob(t).sentiment.subjectivity
    return "Opinion" if v >= 0.5 else "Fact" if v < 0.3 else "Mixed"


def compare(ta, xa, tb, xb, name):
    a = core(ta, xa, name)
    b = core(tb, xb, name)
    if a is None or b is None:
        return MSG_SHORT_BOTH

    def pct(r, k):
        return f"{r[k] / r['n'] * 100:.0f}%"

    rows = [
        ("Headline style", headline_style(ta), headline_style(tb)),
        ("Writing style", a['opinion'], b['opinion']),
        ("Fact sentences", pct(a, 'fact'), pct(b, 'fact')),
        ("Mixed sentences", pct(a, 'mix'), pct(b, 'mix')),
        ("Opinion sentences", pct(a, 'opi'), pct(b, 'opi')),
        ("Strongly worded words",
         esc(', '.join(a['found']) or "none"), esc(', '.join(b['found']) or "none")),
    ]
    if name.strip():
        rows.append((f'Mentions of "{esc(name.strip())}"', name_summary(a), name_summary(b)))

    fa, fb = a['opi'] / a['n'], b['opi'] / b['n']
    if abs(fa - fb) < 0.05:
        line = f"Both articles use a similar share of opinion sentences ({fa:.0%} and {fb:.0%})."
    else:
        more = "A" if fa > fb else "B"
        other = "B" if more == "A" else "A"
        hi, lo = max(fa, fb), min(fa, fb)
        line = f"Article {more} uses more opinion sentences ({hi:.0%} vs {lo:.0%} in Article {other}); the other leans more on facts."

    return (
        f'<div class="cmp-grid">{card_html(a, "Article A", ta)}{card_html(b, "Article B", tb)}</div>'
        f'<div class="summary">{line}</div>{table_html(rows)}'
    )


with gr.Blocks(title="News Sentiment and Style Analyzer") as demo:
    gr.HTML(HEAD_HTML)

    with gr.Tab("Analyze an article"):
        with gr.Column(elem_classes="card"):
            with gr.Row():
                t = gr.Textbox(label="Headline (optional)", lines=1, scale=3)
                nm = gr.Textbox(
                    label="Track a person, party or organization (optional)",
                    placeholder="e.g. Supreme Court, RBI", lines=1, scale=2)
            x = gr.Textbox(label="Article text", lines=12,
                           placeholder="Paste the full article here...")
            btn = gr.Button("\u2728 Analyze article", elem_classes="main-btn")
        with gr.Column(elem_classes="card"):
            o1 = gr.HTML()
            o3 = gr.Plot(label="Tone through the article")
            o4 = gr.Markdown()
        btn.click(single, [t, x, nm], [o1, o3, o4])

    with gr.Tab("Compare two articles"):
        with gr.Column(elem_classes="card"):
            with gr.Row():
                with gr.Column():
                    ta = gr.Textbox(label="Article A headline (optional)", lines=1)
                    xa = gr.Textbox(label="Article A text", lines=10,
                                    placeholder="Paste the first article...")
                with gr.Column():
                    tb = gr.Textbox(label="Article B headline (optional)", lines=1)
                    xb = gr.Textbox(label="Article B text", lines=10,
                                    placeholder="Paste the second article...")
            nm2 = gr.Textbox(
                label="Track a person, party or organization in both (optional)",
                placeholder="e.g. Supreme Court, RBI", lines=1)
            btn2 = gr.Button("\U0001F4AB Compare articles", elem_classes="main-btn")
        with gr.Column(elem_classes="card"):
            c1 = gr.HTML()
        btn2.click(compare, [ta, xa, tb, xb, nm2], [c1])

    with gr.Tab("Write-up"):
        with gr.Column(elem_classes="card"):
            gr.Markdown(WRITEUP, elem_classes="doc")

    with gr.Tab("Limitations"):
        with gr.Column(elem_classes="card"):
            gr.Markdown(LIMITATIONS, elem_classes="doc")

    gr.Markdown(DISCLAIMER, elem_classes="note")
    gr.HTML(FOOT_HTML)

demo.launch()

C:\Users\murth\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
